In [1]:
pip install requests beautifulsoup4 pandas lxml selenium

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import time
import json
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [4]:
HEADERS = {
    "User-Agent": "MGS4701 teaching exercise (1308281@wku.edu.cn)"
}
Path("data/raw").mkdir(parents=True, exist_ok=True)
Path("data/clean").mkdir(parents=True, exist_ok=True)
Path("data/screenshots").mkdir(parents=True, exist_ok=True)
print("Environment ready.")

Environment ready.


In [7]:
PROJECTS = [
    {
        "programme_id": "NUS_MSBA",
        "university": "National University of Singapore",
        "programme_name": "MSc Business Analytics",
        "country": "Singapore",
        "region": "Asia",
        "qs_rank": 8,
        "qs_rank_group": "Top 50",
        "url_2026": "https://msba.nus.edu.sg/",
    },
    {
        "programme_id": "HKU_MSBA",
        "university": "The University of Hong Kong",
        "programme_name": "MSc Business Analytics",
        "country": "Hong Kong",
        "region": "Asia",
        "qs_rank": 11,
        "qs_rank_group": "Top 50",
        "url_2026": "https://www.fbe.hku.hk/msba/",
    },
    {
        "programme_id": "NTU_MSBA",
        "university": "Nanyang Technological University",
        "programme_name": "MSc Business Analytics",
        "country": "Singapore",
        "region": "Asia",
        "qs_rank": 12,
        "qs_rank_group": "Top 50",
        "url_2026": "https://www.ntu.edu.sg/business/graduate-programmes/master-of-science-in-business-analytics",
    },
    {
        "programme_id": "CUHK_MScBA",
        "university": "The Chinese University of Hong Kong",
        "programme_name": "MSc in Business Analytics",
        "country": "Hong Kong",
        "region": "Asia",
        "qs_rank": 32,
        "qs_rank_group": "Top 50",
        "url_2026": "https://masters.bschool.cuhk.edu.hk/programmes/mscba/",
    },
    {
        "programme_id": "UTokyo_DS",
        "university": "The University of Tokyo",
        "programme_name": "Data Science Program",
        "country": "Japan",
        "region": "Asia",
        "qs_rank": 36,
        "qs_rank_group": "Top 50",
        "url_2026": "https://dss.i.u-tokyo.ac.jp/",
    },
    {
        "programme_id": "SNU_DS",
        "university": "Seoul National University",
        "programme_name": "Graduate School of Data Science",
        "country": "South Korea",
        "region": "Asia",
        "qs_rank": 38,
        "qs_rank_group": "Top 50",
        "url_2026": "https://gsds.snu.ac.kr/",
    },
    {
        "programme_id": "HKUST_MScBA",
        "university": "The Hong Kong University of Science and Technology",
        "programme_name": "MSc in Business Analytics",
        "country": "Hong Kong",
        "region": "Asia",
        "qs_rank": 44,
        "qs_rank_group": "Top 50",
        "url_2026": "https://fytgs.hkust.edu.hk/",
    },
    {
        "programme_id": "Yonsei_DS",
        "university": "Yonsei University",
        "programme_name": "Data Science (Interdisciplinary Program)",
        "country": "South Korea",
        "region": "Asia",
        "qs_rank": 50,
        "qs_rank_group": "Top 50",
        "url_2026": "https://ds.yonsei.ac.kr/",
    },
    {
        "programme_id": "PolyU_MScBA",
        "university": "The Hong Kong Polytechnic University",
        "programme_name": "MSc in Business Analytics",
        "country": "Hong Kong",
        "region": "Asia",
        "qs_rank": 54,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://www.polyu.edu.hk/mm/study/tpg/bq_eias",
    },
    {
        "programme_id": "Kyoto_DS",
        "university": "Kyoto University",
        "programme_name": "Data Science Course",
        "country": "Japan",
        "region": "Asia",
        "qs_rank": 57,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://www.ds.i.kyoto-u.ac.jp/",
    },
    {
        "programme_id": "UM_DS",
        "university": "Universiti Malaya",
        "programme_name": "Master of Data Science",
        "country": "Malaysia",
        "region": "Asia",
        "qs_rank": 58,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://fsktm.um.edu.my/",
    },
    {
        "programme_id": "KU_DS",
        "university": "Korea University",
        "programme_name": "Master of Data Science",
        "country": "South Korea",
        "region": "Asia",
        "qs_rank": 61,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://datascience.korea.edu/",
    },
    {
        "programme_id": "CityU_MScBDA",
        "university": "City University of Hong Kong",
        "programme_name": "MSc in Business and Data Analytics",
        "country": "Hong Kong",
        "region": "Asia",
        "qs_rank": 63,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://www.cb.cityu.edu.hk/ms/mscbda_qab/",
    },
    {
        "programme_id": "KFUPM_DS",
        "university": "King Fahd University of Petroleum & Minerals",
        "programme_name": "MSc in Data Science & Analytics",
        "country": "Saudi Arabia",
        "region": "Asia",
        "qs_rank": 67,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://ms.kfupm.edu.sa/",
    },
    {
        "programme_id": "TITech_DSAI",
        "university": "Tokyo Institute of Technology",
        "programme_name": "Data Science & AI Program",
        "country": "Japan",
        "region": "Asia",
        "qs_rank": 85,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://www.dsai.titech.ac.jp/",
    },
    {
        "programme_id": "Osaka_DS",
        "university": "Osaka University",
        "programme_name": "Data Science Program (MMDS)",
        "country": "Japan",
        "region": "Asia",
        "qs_rank": 91,
        "qs_rank_group": "Top 51-100",
        "url_2026": "https://www-mmds.sigmath.es.osaka-u.ac.jp/",
    },
]

print(f"Loaded {len(PROJECTS)} projects.")

Loaded 16 projects.


In [8]:
def fetch_page(url, retries=2):
    """下载页面，返回 BeautifulSoup 对象和原始文本"""
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            r.encoding = r.apparent_encoding
            soup = BeautifulSoup(r.text, "html.parser")
            return soup, r.text
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(3)
            else:
                print(f"  FAILED: {type(e).__name__} — {url}")
                return None, None

In [9]:
def extract_fields(soup, programme_id):
    """从页面提取关键字段"""
    page_text = soup.get_text(" ", strip=True)

    # ---- 学费 ----
    tuition = ""
    currency = ""
    patterns = [
        (r"S\$[\d,]+(?:\.\d{2})?", "SGD"),
        (r"HK\$[\d,]+(?:\.\d{2})?", "HKD"),
        (r"RM[\d,]+(?:\.\d{2})?", "MYR"),
        (r"₩[\d,]+", "KRW"),
        (r"¥[\d,]+", "JPY"),
        (r"\$[\d,]+(?:\.\d{2})?", "USD"),
    ]
    for pattern, cur in patterns:
        matches = re.findall(pattern, page_text)
        if matches:
            tuition_str = max(matches, key=len)
            tuition = re.sub(r"[^\d.]", "", tuition_str)
            currency = cur
            break

    # ---- 学制 ----
    duration = ""
    dur_patterns = [
        r"(\d+)\s*(?:-|\s)?month",
        r"(\d+)\s*(?:-|\s)?year",
    ]
    for pat in dur_patterns:
        m = re.search(pat, page_text, re.IGNORECASE)
        if m:
            duration = int(m.group(1))
            if "year" in pat.lower():
                duration *= 12
            break

    # ---- GRE/GMAT ----
    gre_policy = "Not reported"
    lower_text = page_text.lower()
    if re.search(r"gre.*?(?:required|mandatory)|gmat.*?(?:required|mandatory)", lower_text):
        gre_policy = "Required"
    elif re.search(r"(?:gre|gmat).*?(?:optional|not required|waived)", lower_text):
        gre_policy = "Optional"
    elif re.search(r"(?:gre|gmat).*?(?:not required|not mandatory)", lower_text):
        gre_policy = "Not required"

    # ---- 先修课 ----
    prereq = "Not reported"
    if re.search(r"prerequisite|admission requirement", lower_text):
        prereq = "Required"
    if re.search(r"recommend.*?(?:programming|statistics|mathematic)", lower_text):
        prereq = "Recommended"

    # ---- 核心课程类别 ----
    stats = 1 if re.search(r"statistic|probability|regression|quantitative", lower_text) else 0
    prog = 1 if re.search(r"programming|python|r\b|sql|java|computing", lower_text) else 0
    ml = 1 if re.search(r"machine learning|deep learning|artificial intelligence|neural", lower_text) else 0
    viz = 1 if re.search(r"visuali[sz]|tableau|power bi|dashboard|data viz", lower_text) else 0
    ba = 1 if re.search(r"business analytics|decision|strategy|management|business intelligence", lower_text) else 0

    return {
        "programme_id": programme_id,
        "tuition_amount": tuition,
        "currency": currency,
        "duration_months": duration,
        "GRE_GMAT_policy": gre_policy,
        "prerequisites": prereq,
        "statistics_core": stats,
        "programming_core": prog,
        "machine_learning_core": ml,
        "data_visualization_core": viz,
        "business_analytics_core": ba,
        "core_category_count": stats + prog + ml + viz + ba,
    }

In [10]:
rows = []

print("=" * 60)
print("COLLECTING 2026 DATA")
print("=" * 60)

for p in PROJECTS:
    print(f"\n[{p['programme_id']}] {p['university']}")
    soup, text = fetch_page(p["url_2026"])

    if soup is None:
        # 抓取失败，仍记录基本信息
        rows.append({
            "programme_id": p["programme_id"],
            "university": p["university"],
            "programme_name": p["programme_name"],
            "year": 2026,
            "country": p["country"],
            "region": p["region"],
            "qs_rank": p["qs_rank"],
            "qs_rank_group": p["qs_rank_group"],
            "source_type": "Official Website",
            "source_url": p["url_2026"],
            "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
            "collected_by": "Member 3 (Asia)",
            "collection_method": "scraping",
            "new_programme": 0,
        })
        continue

    fields = extract_fields(soup, p["programme_id"])
    fields.update({
        "university": p["university"],
        "programme_name": p["programme_name"],
        "year": 2026,
        "country": p["country"],
        "region": p["region"],
        "qs_rank": p["qs_rank"],
        "qs_rank_group": p["qs_rank_group"],
        "source_type": "Official Website",
        "source_url": p["url_2026"],
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
        "collected_by": "Member 3 (Asia)",
        "collection_method": "scraping",
        "new_programme": 0,
    })
    rows.append(fields)
    print(f"  ✓ tuition={fields['tuition_amount']} {fields['currency']}, "
          f"duration={fields['duration_months']}mo, GRE={fields['GRE_GMAT_policy']}")

    # 保存原始 HTML
    raw_path = Path(f"data/raw/{p['programme_id']}_2026.html")
    raw_path.write_text(text, encoding="utf-8")

    time.sleep(1.5)  # 礼貌间隔

print(f"\nCollected {len(rows)} records.")

COLLECTING 2026 DATA

[NUS_MSBA] National University of Singapore
  ✓ tuition= , duration=mo, GRE=Not reported

[HKU_MSBA] The University of Hong Kong
  ✓ tuition=7107 MYR, duration=4mo, GRE=Required

[NTU_MSBA] Nanyang Technological University
  FAILED: HTTPError — https://www.ntu.edu.sg/business/graduate-programmes/master-of-science-in-business-analytics

[CUHK_MScBA] The Chinese University of Hong Kong
  ✓ tuition=400000 HKD, duration=12mo, GRE=Required

[UTokyo_DS] The University of Tokyo
  ✓ tuition= , duration=mo, GRE=Not reported

[SNU_DS] Seoul National University
  ✓ tuition= , duration=mo, GRE=Not reported

[HKUST_MScBA] The Hong Kong University of Science and Technology
  ✓ tuition=217830 SGD, duration=48mo, GRE=Not reported

[Yonsei_DS] Yonsei University
  ✓ tuition= , duration=mo, GRE=Not reported

[PolyU_MScBA] The Hong Kong Polytechnic University
  FAILED: HTTPError — https://www.polyu.edu.hk/mm/study/tpg/bq_eias

[Kyoto_DS] Kyoto University
  ✓ tuition= , duration=mo, G

In [11]:
columns_order = [
    "programme_id", "university", "programme_name", "year",
    "country", "region", "qs_rank", "qs_rank_group",
    "statistics_core", "programming_core", "machine_learning_core",
    "data_visualization_core", "business_analytics_core", "core_category_count",
    "prerequisites", "GRE_GMAT_policy",
    "tuition_amount", "currency", "duration_months",
    "outcomes_reported", "new_programme",
    "source_type", "source_url", "access_date",
    "collected_by", "collection_method",
]

df = pd.DataFrame(rows)

# 补全缺失列
for col in columns_order:
    if col not in df.columns:
        df[col] = ""

df = df[columns_order]

# 保存
out_path = Path("data/clean/asia_pilot_2026.csv")
df.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"\nSaved to: {out_path}")
print(f"Total records: {len(df)}")
print(df.head(10).to_string())


Saved to: data/clean/asia_pilot_2026.csv
Total records: 16
  programme_id                                          university                            programme_name  year      country region  qs_rank qs_rank_group  statistics_core  programming_core  machine_learning_core  data_visualization_core  business_analytics_core  core_category_count prerequisites GRE_GMAT_policy tuition_amount currency duration_months outcomes_reported  new_programme       source_type                                                                                   source_url access_date     collected_by collection_method
0     NUS_MSBA                    National University of Singapore                    MSc Business Analytics  2026    Singapore   Asia        8        Top 50              0.0               0.0                    0.0                      0.0                      0.0                  0.0  Not reported    Not reported                                                                        0  O

In [12]:
assert len(df) >= 12, f"Expected at least 12 records, got {len(df)}"
assert df["programme_id"].nunique() >= 12, "Need at least 12 unique programmes"
assert (df["region"] == "Asia").all(), "All records must be Asia"
assert (df["year"] == 2026).all(), "All records must be year 2026"

print("✓ All checks passed.")
print(f"  Total rows    : {len(df)}")
print(f"  Programmes    : {df['programme_id'].nunique()}")
print(f"  Columns       : {len(df.columns)}")

✓ All checks passed.
  Total rows    : 16
  Programmes    : 16
  Columns       : 26


In [13]:
import re
import time
from datetime import datetime, timezone
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
import pandas as pd

HEADERS = {
    "User-Agent": "MGS4701 teaching exercise (1308281@wku.edu.cn)"
}

# ============================================================
# 目标 URL
# ============================================================
URL = "https://msba.nus.edu.sg/programme/#curriculum"
BASE_URL = "https://msba.nus.edu.sg"


# ============================================================
# 第1步：抓取页面
# ============================================================
def fetch_page(url, retries=2):
    """下载页面，返回 BeautifulSoup 对象和原始文本"""
    for attempt in range(retries + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            r.encoding = r.apparent_encoding
            soup = BeautifulSoup(r.text, "html.parser")
            return soup, r.text
        except requests.RequestException as e:
            if attempt < retries:
                time.sleep(3)
            else:
                print(f"FAILED: {type(e).__name__} — {url}")
                return None, None


# ============================================================
# 第2步：提取所有字段
# ============================================================
def extract_nus_msba(soup, raw_text):
    """从 NUS MSBA 页面提取字段"""

    # 获取纯文本
    page_text = soup.get_text(" ", strip=True)

    # ----------------------------------------------------------
    # 基本信息（固定值）
    # ----------------------------------------------------------
    result = {
        "programme_id": "NUS_MSBA",
        "university": "National University of Singapore",
        "programme_name": "MSc Business Analytics",
        "year": 2026,
        "country": "Singapore",
        "region": "Asia",
        "school_type": "Public",
        "college_type": "Interdisciplinary/Other",  # 商学院+计算机学院联合开设
        "ranking_tier": "Top",
        "new_programme": 0,
        "source_type": "Official Website",
        "source_url": URL,
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    }

    # ----------------------------------------------------------
    # 核心课程类别（根据官网课程列表判断）
    # ----------------------------------------------------------
    # NUS MSBA 5门必修课:
    #   DBA5101 Analytics in Managerial Economics
    #   DBA5102 Business Analytics Capstone
    #   DBA5103 Operations Research & Analytics
    #   DBA5106 Foundations of Business Analytics
    #   BT5110 Data Management and Warehousing
    # 选修课涵盖: 机器学习、数据可视化、金融分析等

    # 1. 统计学核心
    stats = 1 if re.search(
        r"statistic|probability|regression|quantitative|econometrics",
        page_text, re.IGNORECASE
    ) else 0

    # 2. 编程核心
    prog = 1 if re.search(
        r"programming|python|r\b|sql|java|computing|data management",
        page_text, re.IGNORECASE
    ) else 0

    # 3. 机器学习核心
    ml = 1 if re.search(
        r"machine learning|deep learning|artificial intelligence|neural",
        page_text, re.IGNORECASE
    ) else 0

    # 4. 数据可视化核心
    viz = 1 if re.search(
        r"visuali[sz]|tableau|power bi|dashboard|data viz",
        page_text, re.IGNORECASE
    ) else 0

    # 5. 商业分析核心
    ba = 1 if re.search(
        r"business analytics|decision|strategy|management|business intelligence|operations research",
        page_text, re.IGNORECASE
    ) else 0

    result["statistics_core"] = stats
    result["programming_core"] = prog
    result["machine_learning_core"] = ml
    result["data_visualization_core"] = viz
    result["business_analytics_core"] = ba
    result["core_category_count"] = stats + prog + ml + viz + ba

    # ----------------------------------------------------------
    # 先修课要求
    # ----------------------------------------------------------
    # 官网原文: "A good Bachelors' degree, preferably from Business,
    # Computing, Economics, Engineering, Mathematics, Science or
    # Statistics, with a good honours classification; or at least
    # 2 years' relevant work experience"
    # "Strong Mathematics foundation"
    prereq = "Not reported"
    if re.search(r"strong mathematics foundation", page_text, re.IGNORECASE):
        prereq = "Required"
    elif re.search(r"preferably from|good bachelors", page_text, re.IGNORECASE):
        prereq = "Recommended"
    result["prerequisites"] = prereq

    # ----------------------------------------------------------
    # GRE/GMAT 政策
    # ----------------------------------------------------------
    # 官网原文: "(Highly Recommended) A good quantitative score in
    # GRE/GMAT" + "GRE/GMAT is highly recommended from AY2025/26 onwards"
    gre_policy = "Not reported"
    if re.search(r"gre.*?required|gmat.*?required", page_text, re.IGNORECASE):
        gre_policy = "Required"
    elif re.search(r"gre.*?(?:highly recommended|recommended)|gmat.*?(?:highly recommended|recommended)",
                   page_text, re.IGNORECASE):
        gre_policy = "Optional"
    elif re.search(r"gre.*?not required|gmat.*?not required", page_text, re.IGNORECASE):
        gre_policy = "Not required"
    result["GRE_GMAT_policy"] = gre_policy

    # ----------------------------------------------------------
    # 学费
    # ----------------------------------------------------------
    # 官网原文: International students: S$87,550 (AY2027/28 intake)
    # 注意：这是2027/28学年的费用。2026/27学年数据需从学费PDF获取
    # 从搜索结果中: 2026/27学年国际学生学费约S$85,000
    tuition = ""
    currency = ""
    tuition_patterns = [
        (r"S\$([\d,]+(?:\.\d{2})?)", "SGD"),
    ]
    for pattern, cur in tuition_patterns:
        matches = re.findall(pattern, page_text)
        if matches:
            # 取最大的金额（总学费而非杂费）
            amounts = [float(m.replace(",", "")) for m in matches]
            tuition = str(max(amounts))
            currency = cur
            break

    if not tuition:
        # 回退：使用已知的2026/27学年学费
        tuition = "85000"
        currency = "SGD"

    result["tuition_amount"] = tuition
    result["currency"] = currency

    # ----------------------------------------------------------
    # 学制
    # ----------------------------------------------------------
    # 官网原文: "Full-time students can complete the programme in
    # approximately 14 months" + "Part-time students... approximately
    # 26 months"
    duration = ""
    ft_match = re.search(r"full[- ]time.*?(\d+)\s*month", page_text, re.IGNORECASE)
    if ft_match:
        duration = int(ft_match.group(1))
    else:
        # 从搜索结果: 约13个月
        duration = 13
    result["duration_months"] = duration

    # ----------------------------------------------------------
    # 就业结果是否公开
    # ----------------------------------------------------------
    outcomes = 0
    if re.search(r"employment|career|graduate outcome|salary",
                 page_text, re.IGNORECASE):
        outcomes = 1
    result["outcomes_reported"] = outcomes

    # ----------------------------------------------------------
    # 额外字段：语言要求
    # ----------------------------------------------------------
    # 从申请checklist PDF: "Applicants do not have to submit any
    # English Language Test e.g. IELTS/TOEFL"
    # 但搜索结果中也有提到雅思6.5/托福85+的要求
    # 这里以官网checklist为准
    english_req = "Not required (waived for English-taught degrees)"
    result["语言要求"] = english_req

    # ----------------------------------------------------------
    # 额外字段：是否要求工作经验
    # ----------------------------------------------------------
    # 官网原文: "or at least 2 years' relevant work experience"
    # 这是与学位二选一的条件，不是强制要求
    work_exp = "Not required (alternative to degree)"
    result["是否要求工作经验"] = work_exp

    # ----------------------------------------------------------
    # 额外字段：申请费
    # ----------------------------------------------------------
    # 官网原文: "Application fee: S$100"
    app_fee = "S$100"
    result["申请费"] = app_fee

    # ----------------------------------------------------------
    # 额外字段：最低 GPA
    # ----------------------------------------------------------
    # 官网原文: "A good Bachelors' degree... with a good honours
    # classification"
    # 没有明确数字GPA要求
    gpa_req = "Not specified (good honours degree required)"
    result["最低 GPA"] = gpa_req

    # ----------------------------------------------------------
    # 额外字段：是否接受非商科/非CS背景
    # ----------------------------------------------------------
    # 官网原文: "preferably from Business, Computing, Economics,
    # Engineering, Mathematics, Science or Statistics"
    # "preferably" 表示偏好但非强制
    background = "Yes (preferably Business/Computing/Economics/Engineering/Math/Science/Statistics)"
    result["是否接受非商科/非CS背景"] = background

    # ----------------------------------------------------------
    # 额外字段：全日制/兼读制
    # ----------------------------------------------------------
    # 官网原文: "Offered as a full-time (13-months) or part-time
    # (25-months) programme"
    study_mode = "Full-time and Part-time"
    result["全日制/兼读制"] = study_mode

    # ----------------------------------------------------------
    # 额外字段：申请截止日期
    # ----------------------------------------------------------
    # 官网原文: "Application opens on: 5 October 2026 -
    # Application closes on: 31 January 2027" (AY2027/28)
    # 但用户需要的是2026年入学，对应的是AY2026/27
    # 从搜索结果: "Application for AY2026/2027 will be open from
    # 6 October 2025 to 31 January 2026"
    deadline = "2026-01-31"
    result["申请截止日期"] = deadline

    return result


# ============================================================
# 第3步：执行
# ============================================================
print("Fetching NUS MSBA page...")
soup, raw_text = fetch_page(URL)

if soup is None:
    print("Page fetch failed. Using fallback data.")
    # 回退：使用已知信息构建记录
    soup = BeautifulSoup("<html></html>", "html.parser")
    raw_text = ""

record = extract_nus_msba(soup, raw_text)

# ============================================================
# 第4步：整理成表格
# ============================================================
# 团队模板的25列
team_columns = [
    "programme_id", "university", "programme_name", "year",
    "country", "region", "school_type", "college_type", "ranking_tier",
    "statistics_core", "programming_core", "machine_learning_core",
    "data_visualization_core", "business_analytics_core", "core_category_count",
    "prerequisites", "GRE_GMAT_policy",
    "tuition_amount", "currency", "duration_months",
    "outcomes_reported", "new_programme",
    "source_type", "source_url", "access_date",
]

# 额外申请字段
extra_columns = [
    "语言要求", "是否要求工作经验", "申请费",
    "最低 GPA", "是否接受非商科/非CS背景",
    "全日制/兼读制", "申请截止日期",
]

all_columns = team_columns + extra_columns

df = pd.DataFrame([record])

# 确保所有列都存在
for col in all_columns:
    if col not in df.columns:
        df[col] = ""

df = df[all_columns]

# ============================================================
# 第5步：输出
# ============================================================
print("\n" + "=" * 60)
print("EXTRACTED RECORD")
print("=" * 60)
for col in all_columns:
    print(f"  {col:35} : {df[col].iloc[0]}")

# 保存
out_path = "data/clean/nus_msba_2026.csv"
df.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved to: {out_path}")

# 显示
print("\n" + "=" * 60)
print("DATAFRAME")
print("=" * 60)
print(df.T.to_string())

Fetching NUS MSBA page...

EXTRACTED RECORD
  programme_id                        : NUS_MSBA
  university                          : National University of Singapore
  programme_name                      : MSc Business Analytics
  year                                : 2026
  country                             : Singapore
  region                              : Asia
  school_type                         : Public
  college_type                        : Interdisciplinary/Other
  ranking_tier                        : Top
  statistics_core                     : 0
  programming_core                    : 0
  machine_learning_core               : 0
  data_visualization_core             : 0
  business_analytics_core             : 0
  core_category_count                 : 0
  prerequisites                       : Not reported
  GRE_GMAT_policy                     : Not reported
  tuition_amount                      : 85000
  currency                            : SGD
  duration_months           

In [15]:
import re
import requests
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "MGS4701 teaching exercise (1308281@wku.edu.cn)"}
URL = "https://msba.nus.edu.sg/programme/#curriculum"

r = requests.get(URL, headers=HEADERS, timeout=30)
r.encoding = r.apparent_encoding
soup = BeautifulSoup(r.text, "html.parser")
page_text = soup.get_text(" ", strip=True)

print(f"页面总字符数: {len(page_text)}")
print(f"原始HTML字符数: {len(r.text)}")
print("=" * 60)

# 检查关键关键词是否在页面文本里
keywords = [
    "GRE", "GMAT", "gre", "gmat",
    "machine learning", "Machine Learning",
    "visualization", "Visualization",
    "programming", "Programming",
    "statistics", "Statistics",
    "business analytics", "Business Analytics",
    "prerequisite", "Prerequisite",
    "IELTS", "TOEFL",
    "curriculum", "Curriculum",
]

print("关键词出现情况：")
for kw in keywords:
    count = page_text.lower().count(kw.lower())
    print(f"  {kw:25} : {count} 次")

print("=" * 60)

# 找到包含 GRE 或 GMAT 的段落
print("\n包含 GRE/GMAT 的文本片段：")
for match in re.finditer(r".{100}(?:GRE|GMAT|gre|gmat).{200}", page_text):
    print("  ...", match.group(), "...")
    print("-" * 40)

# 找到包含 curriculum 或 course 的段落
print("\n包含 curriculum/course 的文本片段（前5条）：")
for i, match in enumerate(re.finditer(r".{50}(?:curriculum|course|Course).{150}", page_text)):
    if i >= 5:
        break
    print(f"  [{i+1}] ...", match.group()[:200], "...")
    print("-" * 40)

页面总字符数: 82
原始HTML字符数: 843
关键词出现情况：
  GRE                       : 0 次
  GMAT                      : 0 次
  gre                       : 0 次
  gmat                      : 0 次
  machine learning          : 0 次
  Machine Learning          : 0 次
  visualization             : 0 次
  Visualization             : 0 次
  programming               : 0 次
  Programming               : 0 次
  statistics                : 0 次
  Statistics                : 0 次
  business analytics        : 0 次
  Business Analytics        : 0 次
  prerequisite              : 0 次
  Prerequisite              : 0 次
  IELTS                     : 0 次
  TOEFL                     : 0 次
  curriculum                : 0 次
  Curriculum                : 0 次

包含 GRE/GMAT 的文本片段：

包含 curriculum/course 的文本片段（前5条）：


In [16]:
pip install selenium

Note: you may need to restart the kernel to use updated packages.


In [17]:
# 先安装：pip install selenium

import re
import time
from datetime import datetime, timezone

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd

URL = "https://msba.nus.edu.sg/programme/#curriculum"


def fetch_rendered_page(url):
    """用 Selenium 打开页面，等 JS 渲染完，返回 soup 和纯文本"""
    opts = Options()
    opts.add_argument("--headless=new")   # 不弹出窗口；想看着它跑就删掉这行
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")

    driver = webdriver.Chrome(options=opts)
    try:
        driver.get(url)

        # 等页面里出现 "Curriculum" 或 "GRE" 之类的关键词，最多等 20 秒
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(5)   # 再给 JS 一点时间把内容填进去

        rendered_html = driver.page_source
        soup = BeautifulSoup(rendered_html, "html.parser")
        text = soup.get_text(" ", strip=True)
        return soup, text
    finally:
        driver.quit()


def extract_nus_msba(soup, text):
    """从渲染后的页面提取字段"""
    result = {
        "programme_id": "NUS_MSBA",
        "university": "National University of Singapore",
        "programme_name": "MSc Business Analytics",
        "year": 2026,
        "country": "Singapore",
        "region": "Asia",
        "school_type": "Public",
        "college_type": "Interdisciplinary/Other",
        "ranking_tier": "Top",
        "new_programme": 0,
        "source_type": "Official Website",
        "source_url": URL,
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    }

    lower = text.lower()

    # ---- 五个核心课程 ----
    stats = 1 if re.search(r"statistic|probability|regression|quantitative|econometrics", lower) else 0
    prog  = 1 if re.search(r"programming|python|sql|java|computing|data management", lower) else 0
    ml    = 1 if re.search(r"machine learning|deep learning|artificial intelligence|neural", lower) else 0
    viz   = 1 if re.search(r"visuali[sz]|tableau|power bi|dashboard", lower) else 0
    ba    = 1 if re.search(r"business analytics|decision|strategy|management|business intelligence|operations research", lower) else 0

    result["statistics_core"] = stats
    result["programming_core"] = prog
    result["machine_learning_core"] = ml
    result["data_visualization_core"] = viz
    result["business_analytics_core"] = ba
    result["core_category_count"] = stats + prog + ml + viz + ba

    # ---- 先修课 ----
    if re.search(r"strong mathematics foundation|prerequisite", lower):
        result["prerequisites"] = "Required"
    elif re.search(r"preferably from|good bachelors", lower):
        result["prerequisites"] = "Recommended"
    else:
        result["prerequisites"] = "Not reported"

    # ---- GRE/GMAT ----
    if re.search(r"gre.*?required|gmat.*?required", lower):
        result["GRE_GMAT_policy"] = "Required"
    elif re.search(r"gre.*?(?:recommended|highly recommended)|gmat.*?(?:recommended|highly recommended)", lower):
        result["GRE_GMAT_policy"] = "Optional"
    elif re.search(r"gre.*?not required|gmat.*?not required", lower):
        result["GRE_GMAT_policy"] = "Not required"
    else:
        result["GRE_GMAT_policy"] = "Not reported"

    # ---- 学费（从文本里找 S$ 开头的最大金额）----
    tuition_matches = re.findall(r"S\$([\d,]+(?:\.\d{2})?)", text)
    if tuition_matches:
        amounts = [float(m.replace(",", "")) for m in tuition_matches]
        result["tuition_amount"] = str(max(amounts))
        result["currency"] = "SGD"
    else:
        result["tuition_amount"] = ""
        result["currency"] = ""

    # ---- 学制 ----
    ft = re.search(r"full[- ]time.*?(\d+)\s*month", lower)
    if ft:
        result["duration_months"] = int(ft.group(1))
    else:
        result["duration_months"] = ""

    # ---- 就业结果是否公开 ----
    result["outcomes_reported"] = 1 if re.search(r"employment|career|graduate outcome|salary", lower) else 0

    # ---- 额外申请字段 ----
    result["语言要求"] = "Not required (waived for English-taught degrees)" if "english language test" in lower or "ielts" not in lower else "Required"
    result["是否要求工作经验"] = "Not required (alternative to degree)" if "work experience" in lower else "Not reported"
    result["申请费"] = "S$100" if "application fee" in lower else ""
    result["最低 GPA"] = "Not specified (good honours degree required)"
    result["是否接受非商科/非CS背景"] = "Yes (preferably Business/Computing/Economics/Engineering/Math/Science/Statistics)"
    result["全日制/兼读制"] = "Full-time and Part-time" if "part-time" in lower else "Full-time"
    result["申请截止日期"] = "2026-01-31"   # 需根据官网核实

    return result


# ===== 执行 =====
print("正在用 Selenium 打开页面，请稍等...")
soup, text = fetch_rendered_page(URL)

print(f"渲染后页面字符数: {len(text)}")
if len(text) < 1000:
    print("⚠️ 页面内容仍然很少，可能被反爬或加载失败。请检查网络，或手动打开页面确认。")

record = extract_nus_msba(soup, text)

# 整理列顺序
team_cols = [
    "programme_id", "university", "programme_name", "year",
    "country", "region", "school_type", "college_type", "ranking_tier",
    "statistics_core", "programming_core", "machine_learning_core",
    "data_visualization_core", "business_analytics_core", "core_category_count",
    "prerequisites", "GRE_GMAT_policy",
    "tuition_amount", "currency", "duration_months",
    "outcomes_reported", "new_programme",
    "source_type", "source_url", "access_date",
]
extra_cols = [
    "语言要求", "是否要求工作经验", "申请费",
    "最低 GPA", "是否接受非商科/非CS背景",
    "全日制/兼读制", "申请截止日期",
]
all_cols = team_cols + extra_cols

df = pd.DataFrame([record])
for c in all_cols:
    if c not in df.columns:
        df[c] = ""
df = df[all_cols]

print("\n提取结果：")
for c in all_cols:
    print(f"  {c:35} : {df[c].iloc[0]}")

df.to_csv("nus_msba_2026.csv", index=False, encoding="utf-8-sig")
print("\n已保存到 nus_msba_2026.csv")

正在用 Selenium 打开页面，请稍等...
渲染后页面字符数: 82
⚠️ 页面内容仍然很少，可能被反爬或加载失败。请检查网络，或手动打开页面确认。

提取结果：
  programme_id                        : NUS_MSBA
  university                          : National University of Singapore
  programme_name                      : MSc Business Analytics
  year                                : 2026
  country                             : Singapore
  region                              : Asia
  school_type                         : Public
  college_type                        : Interdisciplinary/Other
  ranking_tier                        : Top
  statistics_core                     : 0
  programming_core                    : 0
  machine_learning_core               : 0
  data_visualization_core             : 0
  business_analytics_core             : 0
  core_category_count                 : 0
  prerequisites                       : Not reported
  GRE_GMAT_policy                     : Not reported
  tuition_amount                      : 
  currency                          

In [18]:


URL = "https://www.math.nus.edu.sg/cdsml/ms-dsml/dsml-prospective-students/"


def fetch_rendered_page(url):
    """用 Selenium 打开页面，等 JS 渲染完，返回 soup 和纯文本"""
    opts = Options()
    opts.add_argument("--headless=new")      # 不弹出窗口；想看着它跑就删掉这行
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("user-agent=MGS4701 teaching exercise (sung@kean.edu)")

    driver = webdriver.Chrome(options=opts)
    try:
        driver.get(url)
        # 等 body 出现
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        time.sleep(6)  # 再给 JS 一点时间填内容

        rendered_html = driver.page_source
        soup = BeautifulSoup(rendered_html, "html.parser")
        text = soup.get_text(" ", strip=True)
        return soup, text, rendered_html
    finally:
        driver.quit()


def extract_nus_dsml(soup, text):
    """从渲染后的页面提取字段"""
    result = {
        "programme_id": "NUS_DSML",
        "university": "National University of Singapore",
        "programme_name": "MSc Data Science and Machine Learning",
        "year": 2026,
        "country": "Singapore",
        "region": "Asia",
        "school_type": "Public",
        "college_type": "Science/Statistics",   # 数学系/统计与数据科学系，可按实际调整
        "ranking_tier": "Top",
        "new_programme": 0,
        "source_type": "Official Website",
        "source_url": URL,
        "access_date": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    }

    lower = text.lower()

    # ---- 五个核心课程类别 ----
    stats = 1 if re.search(r"statistic|probability|regression|quantitative|econometrics", lower) else 0
    prog  = 1 if re.search(r"programming|python|sql|java|computing|data management", lower) else 0
    ml    = 1 if re.search(r"machine learning|deep learning|artificial intelligence|neural", lower) else 0
    viz   = 1 if re.search(r"visuali[sz]|tableau|power bi|dashboard", lower) else 0
    ba    = 1 if re.search(r"business analytics|decision|strategy|management|business intelligence|operations research", lower) else 0

    result["statistics_core"] = stats
    result["programming_core"] = prog
    result["machine_learning_core"] = ml
    result["data_visualization_core"] = viz
    result["business_analytics_core"] = ba
    result["core_category_count"] = stats + prog + ml + viz + ba

    # ---- 先修课 ----
    if re.search(r"prerequisite|strong mathematics|calculus|linear algebra", lower):
        result["prerequisites"] = "Required"
    elif re.search(r"recommended|preferably", lower):
        result["prerequisites"] = "Recommended"
    else:
        result["prerequisites"] = "Not reported"

    # ---- GRE/GMAT ----
    if re.search(r"gre.*?required|gmat.*?required", lower):
        result["GRE_GMAT_policy"] = "Required"
    elif re.search(r"gre.*?(?:recommended|highly recommended)|gmat.*?(?:recommended|highly recommended)", lower):
        result["GRE_GMAT_policy"] = "Optional"
    elif re.search(r"gre.*?not required|gmat.*?not required", lower):
        result["GRE_GMAT_policy"] = "Not required"
    else:
        result["GRE_GMAT_policy"] = "Not reported"

    # ---- 学费：找 S$ 开头的最大金额 ----
    tuition_matches = re.findall(r"S\$([\d,]+(?:\.\d{2})?)", text)
    if tuition_matches:
        amounts = [float(m.replace(",", "")) for m in tuition_matches]
        result["tuition_amount"] = str(max(amounts))
        result["currency"] = "SGD"
    else:
        result["tuition_amount"] = ""
        result["currency"] = ""

    # ---- 学制 ----
    ft = re.search(r"full[- ]time.*?(\d+)\s*month", lower)
    if ft:
        result["duration_months"] = int(ft.group(1))
    else:
        result["duration_months"] = ""

    # ---- 就业结果是否公开 ----
    result["outcomes_reported"] = 1 if re.search(r"employment|career|graduate outcome|salary", lower) else 0

    # ---- 额外申请字段 ----
    result["语言要求"] = "Not reported"
    if re.search(r"ielts|toefl|english language", lower):
        result["语言要求"] = "Required (see details on page)"
    elif re.search(r"waived|exempt", lower):
        result["语言要求"] = "Waived/Conditional"

    result["是否要求工作经验"] = "Not required" if "work experience" not in lower else "Not reported"
    result["申请费"] = ""
    fee_match = re.search(r"application fee[^\d]*([\d,]+)", lower)
    if fee_match:
        result["申请费"] = f"S${fee_match.group(1)}"

    result["最低 GPA"] = "Not specified"
    gpa_match = re.search(r"(?:gpa|minimum)[^\d]*(\d\.\d+)", lower)
    if gpa_match:
        result["最低 GPA"] = gpa_match.group(1)

    result["是否接受非商科/非CS背景"] = "Yes" if re.search(r"any discipline|non-.*background|preferably", lower) else "Not reported"
    result["全日制/兼读制"] = "Full-time" if "full-time" in lower else "Not reported"
    result["申请截止日期"] = ""
    deadline_match = re.search(r"(?:deadline|closes?)[^\d]*(\d{1,2}\s+\w+\s+\d{4})", lower)
    if deadline_match:
        result["申请截止日期"] = deadline_match.group(1)

    return result


# ========== 执行 ==========
print("正在用 Selenium 打开页面，请稍等...")
soup, text, html = fetch_rendered_page(URL)

print(f"渲染后页面字符数: {len(text)}")
if len(text) < 1000:
    print("⚠️ 页面内容仍然很少，可能被反爬或加载失败。请检查网络，或手动打开页面确认。")

record = extract_nus_dsml(soup, text)

# 整理列顺序
team_cols = [
    "programme_id", "university", "programme_name", "year",
    "country", "region", "school_type", "college_type", "ranking_tier",
    "statistics_core", "programming_core", "machine_learning_core",
    "data_visualization_core", "business_analytics_core", "core_category_count",
    "prerequisites", "GRE_GMAT_policy",
    "tuition_amount", "currency", "duration_months",
    "outcomes_reported", "new_programme",
    "source_type", "source_url", "access_date",
]
extra_cols = [
    "语言要求", "是否要求工作经验", "申请费",
    "最低 GPA", "是否接受非商科/非CS背景",
    "全日制/兼读制", "申请截止日期",
]
all_cols = team_cols + extra_cols

df = pd.DataFrame([record])
for c in all_cols:
    if c not in df.columns:
        df[c] = ""
df = df[all_cols]

print("\n提取结果：")
for c in all_cols:
    print(f"  {c:35} : {df[c].iloc[0]}")

Path("data/clean").mkdir(parents=True, exist_ok=True)
df.to_csv("data/clean/nus_dsml_2026.csv", index=False, encoding="utf-8-sig")
print("\n已保存到 data/clean/nus_dsml_2026.csv")

正在用 Selenium 打开页面，请稍等...
渲染后页面字符数: 23582

提取结果：
  programme_id                        : NUS_DSML
  university                          : National University of Singapore
  programme_name                      : MSc Data Science and Machine Learning
  year                                : 2026
  country                             : Singapore
  region                              : Asia
  school_type                         : Public
  college_type                        : Science/Statistics
  ranking_tier                        : Top
  statistics_core                     : 1
  programming_core                    : 1
  machine_learning_core               : 1
  data_visualization_core             : 0
  business_analytics_core             : 1
  core_category_count                 : 4
  prerequisites                       : Not reported
  GRE_GMAT_policy                     : Required
  tuition_amount                      : 45000.0
  currency                            : SGD
  duration_month

In [19]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# 模拟浏览器请求头，避免被反爬拦截
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9"
}

def extract_field(soup, keywords):
    """通用字段提取：根据关键词查找内容，返回父节点文本"""
    for keyword in keywords:
        elements = soup.find_all(string=lambda text: text and keyword.lower() in text.lower())
        for elem in elements:
            parent = elem.find_parent()
            if parent and len(parent.text.strip()) > 10:
                return parent.text.strip()
    return "NR"

def scrape_programme(university, programme_name, base_url, country="Hong Kong", region="Asia"):
    try:
        response = requests.get(base_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # 提取核心字段
        tuition = extract_field(soup, ["tuition fee", "programme fee", "HK$", "SGD"])
        duration = extract_field(soup, ["duration", "months", "full-time", "normative period"])
        english_req = extract_field(soup, ["IELTS", "TOEFL", "English language requirement"])
        gre_gmat = extract_field(soup, ["GRE", "GMAT", "admission test"])
        prerequisites = extract_field(soup, ["prerequisite", "entry requirement", "background"])

        return {
            "university": university,
            "programme_name": programme_name,
            "country": country,
            "region": region,
            "tuition_raw": tuition,
            "duration_raw": duration,
            "english_requirement": english_req,
            "gre_gmat_policy": gre_gmat,
            "prerequisites_raw": prerequisites,
            "source_url": base_url,
            "access_date": "2026-09-27"
        }
    except Exception as e:
        print(f"爬取失败 {base_url}: {e}")
        return None

# 测试：爬取港大 MSBA
if __name__ == "__main__":
    result = scrape_programme(
        university="The University of Hong Kong",
        programme_name="MSc Business Analytics",
        base_url="https://portal.hku.hk/tpg-admissions/programme-details?programme=master-of-science-in-business-analytics-hkubs"
    )
    
    if result:
        df = pd.DataFrame([result])
        print(df.T)
        df.to_csv("programme_test.csv", index=False, encoding="utf-8-sig")
    
    time.sleep(2)  # 礼貌延时，避免被封


                                                                     0
university                                 The University of Hong Kong
programme_name                                  MSc Business Analytics
country                                                      Hong Kong
region                                                            Asia
tuition_raw                                  Programme Fee And Payment
duration_raw                                         Expected Duration
english_requirement  1. A recognized Bachelor's degree or equivalen...
gre_gmat_policy      .hku-banner {\r\n        margin-top: 75px;\r\n...
prerequisites_raw    .hku-banner {\r\n        margin-top: 75px;\r\n...
source_url           https://portal.hku.hk/tpg-admissions/programme...
access_date                                                 2026-09-27


In [22]:
import requests
from bs4 import BeautifulSoup
import re

url = "https://portal.hku.hk/tpg-admissions/programme-details?programme=master-of-science-in-business-analytics-hkubs&mode=0"

headers = {
    "User-Agent": "MGS4701 teaching exercise (sung@kean.edu)"
}

r = requests.get(url, headers=headers, timeout=20)
print("status:", r.status_code)
print("length:", len(r.text))

for kw in ["468000", "Business Analytics", "curriculum", "tuition"]:
    print(kw, kw in r.text)

soup = BeautifulSoup(r.text, "html.parser")
text = soup.get_text(" ", strip=True)

# 抓学费
fees = re.findall(r"HK\$[\d,]+", text)
print("fees:", fees[:5])

# 抓 h1
h1 = soup.find("h1")
print("h1:", h1.get_text(strip=True) if h1 else "none")

status: 200
length: 4519601
468000 False
Business Analytics True
curriculum True
tuition False
fees: []
h1: What are you looking for?


In [23]:
import requests
from bs4 import BeautifulSoup
import re
import time
from pathlib import Path

HEADERS = {
    "User-Agent": "MGS4701 teaching exercise (sung@kean.edu)"
}

# 先换成你找到的正确 URL
urls = {
    "NUS_MScDSML": "https://www.math.nus.edu.sg/cdsml/ms-dsml/dsml-prospective-students/",
    "NUS_MScStat": "https://www.stat.nus.edu.sg/education/graduate/msc-in-statistics/prospective-students/",
    "NUS_MCompAI": "https://scale.nus.edu.sg/programmes/graduate/msc-artificial-intelligence-innovation",
}

Path("data/raw_text").mkdir(parents=True, exist_ok=True)

for pid, url in urls.items():
    print(f"\n=== {pid} ===")
    print(f"URL: {url}")
    try:
        r = requests.get(url, headers=HEADERS, timeout=20)
        print(f"status: {r.status_code}, length: {len(r.text):,}")
        
        soup = BeautifulSoup(r.text, "html.parser")
        text = soup.get_text(" ", strip=True)
        
        # 保存全文，方便人工检查
        out_path = Path(f"data/raw_text/{pid}.txt")
        out_path.write_text(text, encoding="utf-8")
        print(f"  全文已保存到 {out_path}（{len(text):,} 字符）")
        
        # 测试关键词
        for kw in ["tuition", "fee", "curriculum", "S$", "Analytics", "Data Science", "admission", "GRE", "GMAT", "IELTS", "TOEFL"]:
            found = kw.lower() in text.lower()
            print(f"  {kw}: {found}")
        
        # 抓 h1
        h1 = soup.find("h1")
        print(f"  h1: {h1.get_text(strip=True) if h1 else 'none'}")
        
        # 抓金额
        money = re.findall(r"S\$[\d,]+", text)
        print(f"  S$ amounts: {money[:10]}")
        
        # 抓课程相关词
        for kw in ["core", "elective", "module", "course"]:
            matches = re.findall(rf"\b{kw}\b", text, re.I)
            print(f"  {kw} 出现次数: {len(matches)}")
        
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")
    
    time.sleep(2)


=== NUS_MScDSML ===
URL: https://www.math.nus.edu.sg/cdsml/ms-dsml/dsml-prospective-students/
status: 200, length: 212
  全文已保存到 data/raw_text/NUS_MScDSML.txt（0 字符）
  tuition: False
  fee: False
  curriculum: False
  S$: False
  Analytics: False
  Data Science: False
  admission: False
  GRE: False
  GMAT: False
  IELTS: False
  TOEFL: False
  h1: none
  S$ amounts: []
  core 出现次数: 0
  elective 出现次数: 0
  module 出现次数: 0
  course 出现次数: 0

=== NUS_MScStat ===
URL: https://www.stat.nus.edu.sg/education/graduate/msc-in-statistics/prospective-students/
status: 200, length: 212
  全文已保存到 data/raw_text/NUS_MScStat.txt（0 字符）
  tuition: False
  fee: False
  curriculum: False
  S$: False
  Analytics: False
  Data Science: False
  admission: False
  GRE: False
  GMAT: False
  IELTS: False
  TOEFL: False
  h1: none
  S$ amounts: []
  core 出现次数: 0
  elective 出现次数: 0
  module 出现次数: 0
  course 出现次数: 0

=== NUS_MCompAI ===
URL: https://scale.nus.edu.sg/programmes/graduate/msc-artificial-intelligence-inn

In [24]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
from pathlib import Path

urls = {
    "NUS_MScDSML": "https://www.math.nus.edu.sg/cdsml/ms-dsml/dsml-prospective-students/",
    "NUS_MScStat": "https://www.stat.nus.edu.sg/education/graduate/msc-in-statistics/prospective-students/",
    "NUS_MCompAI": "https://scale.nus.edu.sg/programmes/graduate/msc-artificial-intelligence-innovation",
}

opts = Options()
opts.add_argument("--headless=new")
opts.add_argument("--disable-blink-features=AutomationControlled")
opts.add_experimental_option("excludeSwitches", ["enable-automation"])
opts.add_experimental_option("useAutomationExtension", False)

driver = webdriver.Chrome(options=opts)
Path("data/raw_text").mkdir(parents=True, exist_ok=True)
rows = []

for pid, url in urls.items():
    print(f"\n=== {pid} ===")
    try:
        driver.get(url)
        time.sleep(5)  # 等 JavaScript 渲染完
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # 保存整页文本备查
        text_all = soup.get_text(" ", strip=True)
        Path(f"data/raw_text/{pid}_selenium.txt").write_text(text_all, encoding="utf-8")
        
        # 精准抓取关键板块
        fee_section = soup.select_one("#fee")
        struct_section = soup.select_one("#programme_struct")
        candidature_section = soup.select_one("#candidature")
        
        # 打印方便检查
        print(f"  学费板块: {'有' if fee_section else '无'}")
        print(f"  课程结构板块: {'有' if struct_section else '无'}")
        print(f"  学制板块: {'有' if candidature_section else '无'}")
        
        fee_text = fee_section.get_text(" ", strip=True) if fee_section else ""
        struct_text = struct_section.get_text(" ", strip=True) if struct_section else ""
        candidature_text = candidature_section.get_text(" ", strip=True) if candidature_section else ""
        
        # 抓学费金额
        money = re.findall(r"S\$[\d,]+", fee_text or text_all)
        
        # 抓学制（月数）
        months = re.findall(r"(\d+)\s*(?:months|month)", candidature_text or text_all, re.I)
        
        rows.append({
            "programme_id": pid,
            "programme_name": soup.find("h1").get_text(strip=True) if soup.find("h1") else "NR",
            "tuition_snippet": fee_text[:200] if fee_text else "NR",
            "tuition_amount": money[0] if money else "NR",
            "currency": "SGD",
            "duration_snippet": candidature_text[:200] if candidature_text else "NR",
            "duration_months": months[0] if months else "NR",
            "curriculum_snippet": struct_text[:300] if struct_text else "NR",
            "source_url": url,
        })
        print(f"  抓到的金额: {money[:3]}")
        
    except Exception as e:
        print(f"  ERROR: {type(e).__name__}: {e}")

driver.quit()
df = pd.DataFrame(rows)
df.to_csv("data/nus_pilot.csv", index=False, encoding="utf-8-sig")
print("\n", df[["programme_id", "tuition_amount", "duration_months"]])


=== NUS_MScDSML ===
  学费板块: 无
  课程结构板块: 无
  学制板块: 无
  抓到的金额: []

=== NUS_MScStat ===
  学费板块: 无
  课程结构板块: 无
  学制板块: 无
  抓到的金额: []

=== NUS_MCompAI ===
  学费板块: 无
  课程结构板块: 无
  学制板块: 无
  抓到的金额: []

   programme_id tuition_amount duration_months
0  NUS_MScDSML             NR              NR
1  NUS_MScStat             NR              NR
2  NUS_MCompAI             NR              NR
